In [15]:
import re
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path("197-miniproject")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILES = [
    DATA_DIR / "CONDA_train.csv",
    DATA_DIR / "CONDA_valid.csv",
]

RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

COLUMN_MAP = {
    "matchId": "match_id",
    "conversationId": "conversation_id",
    "playerId": "player_id",
    "playerSlot": "player_slot",
    "chatTime": "chat_time",
    "utterance": "utterance",
    "slotTokens": "slot_tokens",
    "intentClass": "intent_class",
    "slotClasses": "slot_classes",
}

VALID_INTENTS = {"E", "I", "A", "O"}

In [16]:
# Load Data

dfs = []
for path in DATA_FILES:
    if not path.exists():
        raise FileNotFoundError(f"Expected annotated dataset file not found: {path.resolve()}")
    df_part = pd.read_csv(path)
    df_part["source_file"] = path.name
    print(f"Loaded {path.name:<16} | Rows: {len(df_part):>6,}")
    dfs.append(df_part)

raw_df = pd.concat(dfs, ignore_index=True)
raw_df = raw_df.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in raw_df.columns})

print(f"\nCombined annotated dataset shape: {raw_df.shape}")
raw_df.head(3)

Loaded CONDA_train.csv  | Rows: 26,921
Loaded CONDA_valid.csv  | Rows:  8,974

Combined annotated dataset shape: (35895, 11)


,Id,match_id,conversation_id,utterance,chat_time,player_slot,player_id,intent_class,slot_classes,slot_tokens,source_file
0,11263,697,3193,wow!,76,0,ANTS IN MY EYES JOHNSON,O,O,"wow (O),",CONDA_train.csv
1,13741,843,3809,WTF,1563,5,M.k,O,T,"WTF (T),",CONDA_train.csv
2,22125,1412,6199,wpe wpe,2853,1,Acqua Ragia,O,O O,"wpe (O), wpe (O),",CONDA_train.csv


In [ ]:
# Data Cleaning and Sorting

def clean_utterance_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C")
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_df = raw_df.copy()
clean_df["_orig_order"] = np.arange(len(clean_df))

# Drop missing core identifiers, text, or annotations
required_cols = ["match_id", "conversation_id", "chat_time", "utterance", "intent_class"]
clean_df = clean_df.dropna(subset=required_cols).copy()

# Validate utterance-level intent labels (E, I, A, O)
clean_df["intent_class"] = clean_df["intent_class"].astype(str).str.strip().str.upper()
clean_df = clean_df[clean_df["intent_class"].isin(VALID_INTENTS)].copy()

# Clean text and drop empty utterances
clean_df["utterance_clean"] = clean_df["utterance"].apply(clean_utterance_text)
clean_df = clean_df[clean_df["utterance_clean"].str.len() > 0].copy()

# Fill optional metadata fields
clean_df["match_id"] = clean_df["match_id"].astype(str).str.strip()
clean_df["conversation_id"] = clean_df["conversation_id"].astype(str).str.strip()
clean_df["chat_time"] = pd.to_numeric(clean_df["chat_time"], errors="coerce")
clean_df = clean_df.dropna(subset=["chat_time"]).copy()

if "player_id" in clean_df.columns:
    clean_df["player_id"] = clean_df["player_id"].fillna("unknown").astype(str)
if "player_slot" in clean_df.columns:
    clean_df["player_slot"] = clean_df["player_slot"].fillna(-1)
if "slot_tokens" in clean_df.columns:
    clean_df["slot_tokens"] = clean_df["slot_tokens"].fillna("")
if "slot_classes" in clean_df.columns:
    clean_df["slot_classes"] = clean_df["slot_classes"].fillna("")

# Remove exact duplicate utterances that are at the timestamp in the same conversation
clean_df = clean_df.drop_duplicates(
    subset=["match_id", "conversation_id", "chat_time", "utterance_clean", "intent_class"]
).copy()

# Sort messages chronologically per match
clean_df = clean_df.sort_values(
    by=["match_id", "conversation_id", "chat_time", "_orig_order"],
    ascending=[True, True, True, True],
    kind="mergesort",
).reset_index(drop=True)

# Map binary toxicity target (E & I = 1 [Toxic], A & O = 0 [Non-Toxic])
clean_df["toxicity_binary"] = clean_df["intent_class"].isin(["E", "I"]).astype(int)
clean_df["toxicity_label_name"] = clean_df["toxicity_binary"].map({1: "Toxic", 0: "Non-Toxic"})
clean_df["turn_index"] = clean_df.groupby(["match_id", "conversation_id"]).cumcount()
clean_df["char_length"] = clean_df["utterance_clean"].str.len()
clean_df["word_count"] = clean_df["utterance_clean"].str.split().str.len()

clean_df = clean_df.drop(columns=["_orig_order"])
print(f"Usable cleaned records: {len(clean_df):,} (removed {len(raw_df) - len(clean_df):,} unusable rows)")

preview_cols = [
    "match_id",
    "conversation_id",
    "turn_index",
    "chat_time",
    "utterance",
    "utterance_clean",
    "intent_class",
    "toxicity_binary",
    "toxicity_label_name",
]
clean_df[preview_cols].head(10)

Usable cleaned records: 35,683 (removed 212 unusable rows)


,match_id,conversation_id,turn_index,chat_time,utterance,utterance_clean,intent_class,toxicity_binary,toxicity_label_name
0,0,0,0,-8,force it,force it,A,0,Non-Toxic
1,0,0,1,5,space created [SEPA] hah,space created [SEPA] hah,O,0,Non-Toxic
2,0,0,2,9,ez 500,ez 500,I,1,Toxic
3,0,1,0,934,mvp ulti,mvp ulti,O,0,Non-Toxic
4,0,2,0,1488,hah,hah,O,0,Non-Toxic
5,0,3,0,1721,ka bu tooooooooooooo,ka bu tooooooooooooo,O,0,Non-Toxic
6,0,4,0,1854,wtf [SEPA] TA? [SEPA] u srsly?,wtf [SEPA] TA? [SEPA] u srsly?,E,1,Toxic
7,0,4,1,1882,sad spec [SEPA] noes cape for him,sad spec [SEPA] noes cape for him,O,0,Non-Toxic
8,0,4,2,1888,wat [SEPA] that one i cant even run,wat [SEPA] that one i cant even run,O,0,Non-Toxic
9,0,4,3,1905,why alyway hit me [SEPA] what did i do,why alyway hit me [SEPA] what did i do,O,0,Non-Toxic
